In [2]:
%pip install xgboost lightgbm

   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
    --------------------------------------- 1.0/48.9 MB 11.1 MB/s eta 0:00:05
   ---- ----------------------------------- 5.8/48.9 MB 19.3 MB/s eta 0:00:03
   -------- ------------------------------- 10.7/48.9 MB 21.1 MB/s eta 0:00:02
   ------------ --------------------------- 15.7/48.9 MB 22.1 MB/s eta 0:00:02
   ---------------- ----------------------- 20.2/48.9 MB 21.8 MB/s eta 0:00:02
   -------------------- ------------------- 25.7/48.9 MB 22.6 MB/s eta 0:00:02
   ------------------------- -------------- 30.9/48.9 MB 23.1 MB/s eta 0:00:01
   ----------------------------- ---------- 35.9/48.9 MB 23.2 MB/s eta 0:00:01
   --------------------------------- ------ 41.2/48.9 MB 23.5 MB/s eta 0:00:01
   ------------------------------------- -- 46.4/48.9 MB 23.6 MB/s eta 0:00:01
   ---------------------------------------  48.8/48.9 MB 23.6 MB/s eta 0:00:01
   ---------------------------------------  48.8/48.9 MB 23.6 M


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    log_loss,
    brier_score_loss,
    roc_auc_score
)

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import joblib
import matplotlib.pyplot as plt

In [4]:
feature_dir = Path(
    "../../../data/processed/engineered_features/combined_model"
)

model_dir = Path("../../../models/tree_based_models")
model_dir.mkdir(parents=True, exist_ok=True)

numeric_features = [
    "overall_smoothed_accuracy",
    "weighted_same_skill_smoothed_accuracy",
    "log_weighted_same_skill_previous_interactions",
    "weighted_other_skill_smoothed_accuracy",
    "question_smoothed_accuracy",
    "log_question_previous_interactions"
]

categorical_features = [
    "answer_type"
]

feature_columns = numeric_features + categorical_features

target_column = "correct"

key_columns = ["user_id", "order_id"]

In [5]:
fold_datasets = {}

for fold in range(1, 6):

    train_path = feature_dir / f"fold_{fold}_train.csv"
    validation_path = feature_dir / f"fold_{fold}_validation.csv"

    train_df = pd.read_csv(train_path)
    validation_df = pd.read_csv(validation_path)

    fold_datasets[fold] = {
        "train": train_df,
        "validation": validation_df
    }

    print(
        f"Fold {fold}: "
        f"Train = {train_df.shape}, "
        f"Validation = {validation_df.shape}"
    )

Fold 1: Train = (177324, 10), Validation = (42659, 10)
Fold 2: Train = (176273, 10), Validation = (43710, 10)
Fold 3: Train = (170186, 10), Validation = (49797, 10)
Fold 4: Train = (178311, 10), Validation = (41672, 10)
Fold 5: Train = (177838, 10), Validation = (42145, 10)


In [6]:
tree_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

In [7]:
random_forest = Pipeline(
    steps=[
        ("preprocessor", tree_preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=300,
                max_depth=12,
                min_samples_split=10,
                min_samples_leaf=5,
                max_features="sqrt",
                class_weight=None,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

In [8]:
rf_results = []
rf_predictions = []

for fold, datasets in fold_datasets.items():

    print(f"\nTraining Random Forest — Fold {fold}")

    train_df = datasets["train"]
    val_df = datasets["validation"]

    X_train = train_df[feature_columns]
    y_train = train_df[target_column]

    X_val = val_df[feature_columns]
    y_val = val_df[target_column]

    random_forest.fit(X_train, y_train)

    y_pred = random_forest.predict_proba(X_val)[:, 1]

    rf_results.append({
        "fold": fold,
        "log_loss": log_loss(y_val, y_pred),
        "brier_score": brier_score_loss(y_val, y_pred),
        "roc_auc": roc_auc_score(y_val, y_pred)
    })

    rf_predictions.append(pd.DataFrame({
        "user_id": val_df["user_id"].values,
        "order_id": val_df["order_id"].values,
        "fold": fold,
        "y_true": y_val.values,
        "y_pred_proba": y_pred
    }))

    print("Log Loss:", round(rf_results[-1]["log_loss"], 5))
    print("Brier Score:", round(rf_results[-1]["brier_score"], 5))
    print("ROC-AUC:", round(rf_results[-1]["roc_auc"], 5))


Training Random Forest — Fold 1
Log Loss: 0.54583
Brier Score: 0.18318
ROC-AUC: 0.75693

Training Random Forest — Fold 2
Log Loss: 0.54786
Brier Score: 0.18388
ROC-AUC: 0.73537

Training Random Forest — Fold 3
Log Loss: 0.55986
Brier Score: 0.18926
ROC-AUC: 0.74266

Training Random Forest — Fold 4
Log Loss: 0.55056
Brier Score: 0.18517
ROC-AUC: 0.7407

Training Random Forest — Fold 5
Log Loss: 0.53873
Brier Score: 0.18023
ROC-AUC: 0.75743


In [9]:
rf_results_df = pd.DataFrame(rf_results)
rf_oof_df = pd.concat(rf_predictions, ignore_index=True)

print("FOLD-WISE SUMMARY")
display(
    rf_results_df[
        ["log_loss", "brier_score", "roc_auc"]
    ].agg(["mean", "std"])
)

print("\nPOOLED OOF PERFORMANCE")

print("Log Loss:", log_loss(
    rf_oof_df["y_true"],
    rf_oof_df["y_pred_proba"]
))

print("Brier Score:", brier_score_loss(
    rf_oof_df["y_true"],
    rf_oof_df["y_pred_proba"]
))

print("ROC-AUC:", roc_auc_score(
    rf_oof_df["y_true"],
    rf_oof_df["y_pred_proba"]
))

print("\nTotal predictions:", len(rf_oof_df))
print(
    "Duplicate interactions:",
    rf_oof_df.duplicated(["user_id", "order_id"]).sum()
)

FOLD-WISE SUMMARY


,log_loss,brier_score,roc_auc
mean,0.548569,0.184344,0.746617
std,0.007688,0.003290,0.010003



POOLED OOF PERFORMANCE
Log Loss: 0.548945924392964
Brier Score: 0.18450732415574445
ROC-AUC: 0.746811173619811

Total predictions: 219983
Duplicate interactions: 0


### XG BOOST

In [10]:
xgb_model = Pipeline(
    steps=[
        ("preprocessor", tree_preprocessor),
        (
            "classifier",
            XGBClassifier(
                objective="binary:logistic",
                eval_metric="logloss",
                n_estimators=500,
                learning_rate=0.03,
                max_depth=5,
                min_child_weight=5,
                subsample=0.8,
                colsample_bytree=0.8,
                reg_lambda=5,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

In [11]:
xgb_results = []
xgb_predictions = []

for fold, datasets in fold_datasets.items():

    print(f"\nTraining XGBoost — Fold {fold}")

    train_df = datasets["train"]
    val_df = datasets["validation"]

    X_train = train_df[feature_columns]
    y_train = train_df[target_column]

    X_val = val_df[feature_columns]
    y_val = val_df[target_column]

    xgb_model.fit(X_train, y_train)

    y_pred = xgb_model.predict_proba(X_val)[:, 1]

    xgb_results.append({
        "fold": fold,
        "log_loss": log_loss(y_val, y_pred),
        "brier_score": brier_score_loss(y_val, y_pred),
        "roc_auc": roc_auc_score(y_val, y_pred)
    })

    xgb_predictions.append(pd.DataFrame({
        "user_id": val_df["user_id"].values,
        "order_id": val_df["order_id"].values,
        "fold": fold,
        "y_true": y_val.values,
        "y_pred_proba": y_pred
    }))

    print("Log Loss:", round(xgb_results[-1]["log_loss"], 5))
    print("Brier Score:", round(xgb_results[-1]["brier_score"], 5))
    print("ROC-AUC:", round(xgb_results[-1]["roc_auc"], 5))


Training XGBoost — Fold 1
Log Loss: 0.54352
Brier Score: 0.18235
ROC-AUC: 0.75862

Training XGBoost — Fold 2
Log Loss: 0.54715
Brier Score: 0.1837
ROC-AUC: 0.73588

Training XGBoost — Fold 3
Log Loss: 0.55844
Brier Score: 0.18881
ROC-AUC: 0.74323

Training XGBoost — Fold 4
Log Loss: 0.54967
Brier Score: 0.18486
ROC-AUC: 0.74113

Training XGBoost — Fold 5
Log Loss: 0.53667
Brier Score: 0.1795
ROC-AUC: 0.7588


In [12]:
xgb_results_df = pd.DataFrame(xgb_results)
xgb_oof_df = pd.concat(xgb_predictions, ignore_index=True)

print("FOLD-WISE SUMMARY")

display(
    xgb_results_df[
        ["log_loss", "brier_score", "roc_auc"]
    ].agg(["mean", "std"])
)

print("\nPOOLED OOF PERFORMANCE")

print("Log Loss:", log_loss(
    xgb_oof_df["y_true"],
    xgb_oof_df["y_pred_proba"]
))

print("Brier Score:", brier_score_loss(
    xgb_oof_df["y_true"],
    xgb_oof_df["y_pred_proba"]
))

print("ROC-AUC:", roc_auc_score(
    xgb_oof_df["y_true"],
    xgb_oof_df["y_pred_proba"]
))

print("\nTotal predictions:", len(xgb_oof_df))

print(
    "Duplicate interactions:",
    xgb_oof_df.duplicated(["user_id", "order_id"]).sum()
)

FOLD-WISE SUMMARY


,log_loss,brier_score,roc_auc
mean,0.547091,0.183847,0.747532
std,0.008014,0.003421,0.010550



POOLED OOF PERFORMANCE
Log Loss: 0.5474720597267151
Brier Score: 0.1840127557516098
ROC-AUC: 0.7477293894536783

Total predictions: 219983
Duplicate interactions: 0


In [13]:
lgbm_model = Pipeline(
    steps=[
        ("preprocessor", tree_preprocessor),
        (
            "classifier",
            LGBMClassifier(
                objective="binary",
                n_estimators=500,
                learning_rate=0.03,
                max_depth=7,
                num_leaves=20,
                min_child_samples=30,
                subsample=0.8,
                colsample_bytree=0.8,
                reg_lambda=5,
                random_state=42,
                n_jobs=-1,
                verbosity=-1
            )
        )
    ]
)

In [14]:
lgbm_results = []
lgbm_predictions = []

for fold, datasets in fold_datasets.items():

    print(f"\nTraining LightGBM — Fold {fold}")

    train_df = datasets["train"]
    val_df = datasets["validation"]

    X_train = train_df[feature_columns]
    y_train = train_df[target_column]

    X_val = val_df[feature_columns]
    y_val = val_df[target_column]

    lgbm_model.fit(X_train, y_train)

    y_pred = lgbm_model.predict_proba(X_val)[:, 1]

    lgbm_results.append({
        "fold": fold,
        "log_loss": log_loss(y_val, y_pred),
        "brier_score": brier_score_loss(y_val, y_pred),
        "roc_auc": roc_auc_score(y_val, y_pred)
    })

    lgbm_predictions.append(pd.DataFrame({
        "user_id": val_df["user_id"].values,
        "order_id": val_df["order_id"].values,
        "fold": fold,
        "y_true": y_val.values,
        "y_pred_proba": y_pred
    }))

    print("Log Loss:", round(lgbm_results[-1]["log_loss"], 5))
    print("Brier Score:", round(lgbm_results[-1]["brier_score"], 5))
    print("ROC-AUC:", round(lgbm_results[-1]["roc_auc"], 5))


Training LightGBM — Fold 1
Log Loss: 0.54403
Brier Score: 0.18254
ROC-AUC: 0.75811

Training LightGBM — Fold 2
Log Loss: 0.54759
Brier Score: 0.18387
ROC-AUC: 0.73544

Training LightGBM — Fold 3
Log Loss: 0.55868
Brier Score: 0.18893
ROC-AUC: 0.7429

Training LightGBM — Fold 4
Log Loss: 0.55017
Brier Score: 0.18505
ROC-AUC: 0.74068

Training LightGBM — Fold 5
Log Loss: 0.53708
Brier Score: 0.17967
ROC-AUC: 0.75847


In [15]:
lgbm_results_df = pd.DataFrame(lgbm_results)
lgbm_oof_df = pd.concat(lgbm_predictions, ignore_index=True)

print("FOLD-WISE SUMMARY")

display(
    lgbm_results_df[
        ["log_loss", "brier_score", "roc_auc"]
    ].agg(["mean", "std"])
)

print("\nPOOLED OOF PERFORMANCE")

print("Log Loss:", log_loss(
    lgbm_oof_df["y_true"],
    lgbm_oof_df["y_pred_proba"]
))

print("Brier Score:", brier_score_loss(
    lgbm_oof_df["y_true"],
    lgbm_oof_df["y_pred_proba"]
))

print("ROC-AUC:", roc_auc_score(
    lgbm_oof_df["y_true"],
    lgbm_oof_df["y_pred_proba"]
))

FOLD-WISE SUMMARY


,log_loss,brier_score,roc_auc
mean,0.547512,0.184010,0.747121
std,0.007948,0.003401,0.010551



POOLED OOF PERFORMANCE
Log Loss: 0.547886940352018
Brier Score: 0.18417421421311062
ROC-AUC: 0.7473070531791368


In [16]:
prediction_dir = Path(
    "../../../data/processed/model_predictions"
)
prediction_dir.mkdir(parents=True, exist_ok=True)

tree_model_predictions = {
    "random_forest": rf_oof_df,
    "xgboost": xgb_oof_df,
    "lightgbm": lgbm_oof_df
}

for model_name, predictions in tree_model_predictions.items():

    output_path = (
        prediction_dir / f"{model_name}_predictions.csv"
    )

    predictions.to_csv(output_path, index=False)

    print(f"Saved: {output_path}")

Saved: ..\..\..\data\processed\model_predictions\random_forest_predictions.csv
Saved: ..\..\..\data\processed\model_predictions\xgboost_predictions.csv
Saved: ..\..\..\data\processed\model_predictions\lightgbm_predictions.csv


In [17]:
# Load Logistic Regression OOF predictions
lr_oof_df = pd.read_csv(
    prediction_dir / "combined_model_predictions.csv"
)

all_model_predictions = {
    "Logistic Regression": lr_oof_df,
    "Random Forest": rf_oof_df,
    "XGBoost": xgb_oof_df,
    "LightGBM": lgbm_oof_df
}

comparison_results = []

for model_name, predictions in all_model_predictions.items():

    comparison_results.append({
        "model": model_name,
        "log_loss": log_loss(
            predictions["y_true"],
            predictions["y_pred_proba"]
        ),
        "brier_score": brier_score_loss(
            predictions["y_true"],
            predictions["y_pred_proba"]
        ),
        "roc_auc": roc_auc_score(
            predictions["y_true"],
            predictions["y_pred_proba"]
        )
    })

comparison_df = pd.DataFrame(comparison_results)

comparison_df = comparison_df.sort_values(
    "log_loss"
).reset_index(drop=True)

display(comparison_df)

,model,log_loss,brier_score,roc_auc
0,XGBoost,0.547472,0.184013,0.747729
1,LightGBM,0.547887,0.184174,0.747307
2,Random Forest,0.548946,0.184507,0.746811
3,Logistic Regression,0.554854,0.186792,0.739631
